# NaijaReviewer-8B — End-to-End Pipeline v3 (NVIDIA Data Designer + Unsloth)

**One notebook. Five parts. NVIDIA Data Designer for synthetic data + Unsloth for training. Drive-persisted. Auto-resumes after disconnects.**

| Part | What | Tool | Time (A100 40GB) |
|---|---|---|---|
| 1 | Build ~20k Nigerian fine-tune corpus | **NVIDIA Data Designer** SDK | ~1–2 hr |
| 2 | LLM-as-Judge quality filter | data_designer + Llama 3.3 70B | ~10 min |
| 3 | QLoRA fine-tune Llama 3.1 8B → NaijaReviewer-8B | Unsloth (2.4× faster than vanilla TRL) | ~50 min |
| 4 | Smoke test + HF push | Unsloth + Anthropic | ~10 min |
| 5 | GGUF conversion notes for Ollama | — | local |

## What changed from v2

We replaced our raw `httpx` calls to NVIDIA NIM (which hit 429 rate-limits at concurrency 12) with the **NVIDIA Data Designer SDK** (`pip install data-designer`). The SDK handles:
- **Rate limiting** natively (internal queue + backoff) — no more 429 avalanches
- **Higher effective parallelism** via tuned `buffer_size` and `max_parallel_requests`
- **Declarative pipeline config** — categorical samplers, LLM columns, judge columns, seed datasets
- **Per-batch checkpointing** at 1000-row granularity
- **LLM-as-Judge quality scoring** — filter low-quality samples before fine-tune

## Inputs needed

- `NVIDIA_API_KEY` — from [build.nvidia.com](https://build.nvidia.com)
- `ANTHROPIC_API_KEY` — used only in eval baseline (optional)
- `HF_TOKEN` — for model + dataset push + Llama 3.1 8B download (accept license at [meta-llama/Meta-Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct))

## Resumability

**Just re-run the notebook from the top.** Corpus stages skip per-batch checkpoints that already exist on Drive. Training resumes from the last `checkpoint-XXXX`.

## 0. Setup — Drive + Repo + Deps + Secrets + GPU Autotune

In [ ]:
# Mount Drive
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_DIR = Path("/content/drive/MyDrive/naija-persona-agent")
else:
    DRIVE_DIR = Path.home() / "naija-persona-agent-runs"

DRIVE_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_DIR    = DRIVE_DIR / "data" / "finetune"
CKPT_DIR_ROOT = CORPUS_DIR / "checkpoints"
P1_CKPT_DIR   = CKPT_DIR_ROOT / "pipeline1_seed_jumia"
P2_CKPT_DIR   = CKPT_DIR_ROOT / "pipeline2_synthetic"
P3_CKPT_DIR   = CKPT_DIR_ROOT / "pipeline3_judge"
TRAINING_CKPT = DRIVE_DIR / "checkpoints" / "naija-reviewer-v1"
MERGED_DIR    = DRIVE_DIR / "merged" / "naija-reviewer-8b"
RESULTS_DIR   = DRIVE_DIR / "results"

for d in (CORPUS_DIR, CKPT_DIR_ROOT, P1_CKPT_DIR, P2_CKPT_DIR, P3_CKPT_DIR,
          TRAINING_CKPT, MERGED_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ Drive root: {DRIVE_DIR}")
print(f"   pipelines: P1={P1_CKPT_DIR.name} / P2={P2_CKPT_DIR.name} / P3={P3_CKPT_DIR.name}")

In [ ]:
# Clone the repo
REPO_URL = "https://github.com/Mystique1337/telcoproject"

if IN_COLAB:
    REPO_DIR = Path("/content/telcoproject")
    if REPO_DIR.exists() and not (REPO_DIR / "app").exists():
        import shutil; shutil.rmtree(REPO_DIR)
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull origin main 2>&1 || true
    %cd {REPO_DIR}
    if not (REPO_DIR / "app").exists():
        raise RuntimeError(f"{REPO_URL} appears empty — push from local first.")
else:
    if not (Path.cwd() / "app").exists():
        for c in [Path.cwd().parent, Path.cwd() / "telcoproject"]:
            if (c / "app").exists():
                os.chdir(c); break
    REPO_DIR = Path.cwd()

assert (Path.cwd() / "app").exists(), "Couldn't locate repo root"
print(f"  Repo: {REPO_DIR}")

In [ ]:
# Dependencies
%pip install -q \
    data-designer jsonlines \
    pandas "pyarrow>=19.0.1,<20" \
    tqdm nest_asyncio \
    pydantic==2.9.* pydantic-settings==2.6.* jinja2 chromadb==0.5.* \
    anthropic==0.39.* datasets==3.1.*

# Unsloth — for training (Part 3)
%pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q --no-deps "trl<0.13" peft accelerate bitsandbytes wandb sentencepiece

print("\n✅ Deps installed")

In [ ]:
# Secrets
def _set_secret(name: str, prompt: str):
    if IN_COLAB:
        from google.colab import userdata
        try:
            v = userdata.get(name)
            if v:
                os.environ[name] = v
                print(f"  ✅ {name} from Colab Secrets")
                return v
        except Exception: pass
    v = os.environ.get(name)
    if not v:
        from getpass import getpass
        v = getpass(prompt); os.environ[name] = v
    print(f"  ✅ {name} configured")
    return v

_set_secret("NVIDIA_API_KEY",    "NVIDIA_API_KEY (nvapi-…): ")
_set_secret("ANTHROPIC_API_KEY", "ANTHROPIC_API_KEY (sk-ant-…): ")
_set_secret("HF_TOKEN",          "HF_TOKEN (hf_…): ")

In [ ]:
# nest_asyncio for async-in-notebook
sys.path.insert(0, str(REPO_DIR))
import nest_asyncio; nest_asyncio.apply()
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s — %(message)s")
logger = logging.getLogger("corpus")

In [ ]:
# GPU autotune
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU — Runtime → Change runtime type")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16_OK = torch.cuda.is_bf16_supported()

name_l = GPU_NAME.lower()
if "h100" in name_l:               GPU_CFG = {"batch":16, "grad_accum":2,  "workers":4, "seq_len":4096}
elif "a100" in name_l and GPU_VRAM>60: GPU_CFG = {"batch":16, "grad_accum":2,  "workers":4, "seq_len":4096}
elif "a100" in name_l:             GPU_CFG = {"batch":8,  "grad_accum":4,  "workers":4, "seq_len":4096}
elif "l4" in name_l:               GPU_CFG = {"batch":4,  "grad_accum":8,  "workers":2, "seq_len":3072}
elif "t4" in name_l:               GPU_CFG = {"batch":2,  "grad_accum":16, "workers":2, "seq_len":2048}
else:                              GPU_CFG = {"batch":4,  "grad_accum":8,  "workers":2, "seq_len":3072}

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print(f"GPU: {GPU_NAME} ({GPU_VRAM:.1f}GB) bf16={BF16_OK}")
print(f"Autotuned: batch={GPU_CFG['batch']}, grad_accum={GPU_CFG['grad_accum']}, seq_len={GPU_CFG['seq_len']}")

## 1. Initialize NVIDIA Data Designer

In [ ]:
# Generation settings
P1_TARGET   = 12000  # seed-grounded from Jumia reviews
P2_TARGET   =  8000  # pure synthetic persona × product × register
BATCH_SIZE  =  1000  # auto-save every N records to Drive
JUDGE_SAMPLE_PCT  = 0.20
QUALITY_THRESHOLD = 3.0

# Models — verified working on user's NIM tier
PRIMARY_MODEL = "meta/llama-3.3-70b-instruct"
FAST_MODEL    = "nvidia/llama-3.1-nemotron-nano-8b-v1"
JUDGE_MODEL   = "meta/llama-3.3-70b-instruct"

print(f"Target: {P1_TARGET + P2_TARGET:,} total examples (P1={P1_TARGET:,} + P2={P2_TARGET:,})")
print(f"Models: primary={PRIMARY_MODEL}, fast={FAST_MODEL}, judge={JUDGE_MODEL}")

In [ ]:
# Initialize Data Designer with tuned RunConfig + custom ModelConfigs
import data_designer.config as dd
from data_designer.interface import DataDesigner

run_config = dd.RunConfig(
    buffer_size=2000,
    non_inference_max_parallel_workers=8,
    max_conversation_restarts=5,
    max_conversation_correction_steps=1,
    disable_early_shutdown=True,
)

data_designer = DataDesigner()
data_designer.set_run_config(run_config)

MODEL_PRIMARY = dd.ModelConfig(
    alias="naija-primary",
    model=PRIMARY_MODEL, provider="nvidia",
    inference_parameters=dd.ChatCompletionInferenceParams(
        max_parallel_requests=8, temperature=0.75, top_p=0.95, max_tokens=600,
    ),
)
MODEL_FAST = dd.ModelConfig(
    alias="naija-fast",
    model=FAST_MODEL, provider="nvidia",
    inference_parameters=dd.ChatCompletionInferenceParams(
        max_parallel_requests=12, temperature=0.6, top_p=0.95, max_tokens=400,
    ),
)
MODEL_JUDGE = dd.ModelConfig(
    alias="naija-judge",
    model=JUDGE_MODEL, provider="nvidia",
    inference_parameters=dd.ChatCompletionInferenceParams(
        max_parallel_requests=8, temperature=0.2, top_p=0.95, max_tokens=300,
    ),
)

print("✅ Data Designer initialised")
print(f"   buffer_size={run_config.buffer_size}, primary parallel={MODEL_PRIMARY.inference_parameters.max_parallel_requests}")

In [ ]:
# Batched generation helper — auto-save + resume from Drive
import time, traceback
import pandas as pd
import jsonlines

def count_existing_checkpoints(ckpt_dir):
    files = sorted(ckpt_dir.glob("batch_*.parquet"))
    total = 0
    for f in files:
        try: total += len(pd.read_parquet(f))
        except: pass
    return len(files), total, files

def _load_all_checkpoints(ckpt_dir):
    files = sorted(ckpt_dir.glob("batch_*.parquet"))
    if not files: return pd.DataFrame()
    dfs = []
    for f in files:
        try: dfs.append(pd.read_parquet(f))
        except Exception as e: print(f"  ⚠️ corrupted {f.name}: {e}")
    out = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    print(f"  📦 loaded {len(out):,} from {len(dfs)} checkpoints")
    return out

def generate_batched(config_builder, total_records, batch_size, ckpt_dir, dataset_name):
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    done_batches, done_records, _ = count_existing_checkpoints(ckpt_dir)

    if done_records >= total_records:
        print(f"✅ {dataset_name}: already {done_records:,} on Drive")
        return _load_all_checkpoints(ckpt_dir)

    if done_records > 0:
        print(f"🔄 {dataset_name}: resuming from batch {done_batches+1} ({done_records:,} done)")
    else:
        print(f"🚀 {dataset_name}: starting fresh ({total_records:,} target)")

    batch_num = done_batches + 1
    generated = done_records
    t_start = time.time()

    while generated < total_records:
        this_n = min(batch_size, total_records - generated)
        t_batch = time.time()
        print(f"\n📦 batch {batch_num} — {this_n:,} records ({100*generated/total_records:.1f}% so far)")
        try:
            results = data_designer.create(
                config_builder=config_builder, num_records=this_n,
                dataset_name=f"{dataset_name}_b{batch_num:03d}",
            )
            df = results.load_dataset()
            if df is None or len(df) == 0:
                print("  ⚠️ empty; retrying in 10s"); time.sleep(10); continue

            p = ckpt_dir / f"batch_{batch_num:03d}.parquet"
            df.to_parquet(p, index=False)
            elapsed = time.time() - t_batch
            generated += len(df)
            rate = len(df) / elapsed * 60 if elapsed else 0
            print(f"  ✅ {len(df):,} in {elapsed:.0f}s ({rate:.0f}/min) → {p.name}")

            sess_done = generated - done_records
            if sess_done > 0:
                eta = (total_records - generated) * (time.time() - t_start) / sess_done / 60
                print(f"  ⏱️ ETA: ~{eta:.0f} min")
            batch_num += 1
        except KeyboardInterrupt:
            print(f"\n⏸️ interrupted at {generated:,} — re-run to resume"); break
        except Exception as e:
            print(f"  ❌ {type(e).__name__}: {e}")
            traceback.print_exc()
            print("  ⏳ retry in 30s"); time.sleep(30); continue

    print(f"\n✅ session in {(time.time()-t_start)/60:.1f}min")
    return _load_all_checkpoints(ckpt_dir)

print("✅ batched-generation helpers ready")

## 2. Prepare Seed Data on Drive

In [ ]:
# Download and cache seed datasets to Drive
import httpx, csv, json

JUMIA_CSV_URL = (
    "https://raw.githubusercontent.com/"
    "aymane-maghouti/Sentiment-Analysis-for-Jumia-Reviews-and-Smartphone-Price-Prediction-System/"
    "main/Main/Sentiment_Analysis_for_Jumia_Reviews/final_data.csv"
)
SEED_JUMIA_PATH    = CORPUS_DIR / "seed_jumia_reviews.parquet"
SEED_PRODUCTS_PATH = CORPUS_DIR / "seed_jumia_products.parquet"

# Real Jumia reviews
if SEED_JUMIA_PATH.exists():
    df_jumia = pd.read_parquet(SEED_JUMIA_PATH)
    print(f"✅ Jumia reviews: {len(df_jumia):,} rows on Drive")
else:
    print("📥 downloading aymane Jumia CSV (~15MB)...")
    raw_csv = CORPUS_DIR / "raw_jumia.csv"
    with httpx.Client(timeout=120.0) as c:
        r = c.get(JUMIA_CSV_URL); r.raise_for_status()
        raw_csv.write_bytes(r.content)
    rows = []
    with raw_csv.open(encoding="utf-8", errors="replace") as fh:
        rd = csv.reader(fh); next(rd, None)
        for r in rd:
            if len(r) < 2: continue
            text = " ".join(r[0].split()).strip()
            if not (50 <= len(text) <= 1500): continue
            try: tgt = int(r[1])
            except: continue
            rows.append({"review_text": text, "binary_sentiment": tgt})
    df_jumia = pd.DataFrame(rows)
    df_jumia.to_parquet(SEED_JUMIA_PATH, index=False)
    print(f"✅ saved {len(df_jumia):,} Jumia reviews → {SEED_JUMIA_PATH}")

# Jumia product catalog
if SEED_PRODUCTS_PATH.exists():
    df_products = pd.read_parquet(SEED_PRODUCTS_PATH)
    print(f"✅ Jumia products: {len(df_products):,} rows on Drive")
else:
    print("📥 downloading Idowenst Jumia catalog from HF...")
    from datasets import load_dataset
    ds = load_dataset("Idowenst/jumia_dataset", split="train")
    df_products = pd.DataFrame([
        {"title": r.get("name",""), "category": r.get("category",""),
         "description": (r.get("description") or "")[:600], "price_naira": r.get("price")}
        for r in ds if r.get("name")
    ])
    df_products.to_parquet(SEED_PRODUCTS_PATH, index=False)
    print(f"✅ saved {len(df_products):,} products → {SEED_PRODUCTS_PATH}")

## 3. Pipeline 1 — Seed-Grounded: real Jumia review → (persona, rating, register, rewritten review)

In [ ]:
# Pipeline 1 config
config_p1 = dd.DataDesignerConfigBuilder()
config_p1.add_model_config(MODEL_PRIMARY)
config_p1.with_seed_dataset(
    seed_source={"seed_type": "local", "path": str(SEED_JUMIA_PATH)},
    sampling_strategy="shuffle",
)

config_p1.add_column(dd.SamplerColumnConfig(
    name="persona_id", sampler_type=dd.SamplerType.CATEGORY,
    params=dd.CategorySamplerParams(values=["chinwe_owerri", "tunde_lagos", "aisha_kano", "femi_abuja", "ifeoma_ph"]),
))
config_p1.add_column(dd.SamplerColumnConfig(
    name="register_tier", sampler_type=dd.SamplerType.CATEGORY,
    params=dd.CategorySamplerParams(values=[
        "nigerian_pidgin", "nigerian_pidgin",
        "code_mixed", "code_mixed",
        "nigerian_english", "standard_english",
    ]),
))
config_p1.add_column(dd.SamplerColumnConfig(
    name="product_category", sampler_type=dd.SamplerType.CATEGORY,
    params=dd.CategorySamplerParams(values=[
        "phone_tablet", "electronics", "appliances", "computing",
        "health_beauty", "fashion", "home_office", "baby_products",
        "sports_outdoor", "automobile",
    ]),
))

config_p1.add_column(dd.LLMTextColumnConfig(
    name="refined_rating", model_alias="naija-primary",
    system_prompt=(
        "You calibrate Nigerian product reviewers. Given a Jumia review and its binary "
        "sentiment (+1 pos / -1 neg), output ONLY a single digit 1-5 matching the text's intensity."
    ),
    prompt=(
        "Review: {{ review_text }}\n\n"
        "Binary sentiment: {{ binary_sentiment }}\n\n"
        "1 = strongly negative ('rubbish', 'scam')\n"
        "2 = clearly negative\n"
        "3 = mixed/neutral with caveats\n"
        "4 = clearly positive\n"
        "5 = strongly positive ('e too much', 'scatter', 'best ever')\n\n"
        "Output ONLY one digit (1, 2, 3, 4, or 5)."
    ),
))

config_p1.add_column(dd.LLMTextColumnConfig(
    name="persona_review", model_alias="naija-primary",
    system_prompt=(
        "You rewrite real Nigerian product reviews in the voice of specific persona archetypes, "
        "preserving sentiment and content but matching the user's register and framing. "
        "Output ONLY the rewritten review text. No labels, no quotes, no preamble."
    ),
    prompt=(
        "ORIGINAL: {{ review_text }}\n\n"
        "PERSONA: {{ persona_id }}\n"
        "REGISTER: {{ register_tier }}\n"
        "CATEGORY: {{ product_category }}\n"
        "RATING: {{ refined_rating }}/5\n\n"
        "Rewrite the review:\n"
        "- Keep the SAME sentiment and key points\n"
        "- Match the persona's voice + register strictly\n"
        "- 2-5 sentences, sound real (no AI tells)\n\n"
        "PERSONAS:\n"
        "chinwe_owerri: Owerri Gen-Z, Igbo+English code-mix (nna, biko), communal-hedonic.\n"
        "tunde_lagos: Lagos market trader, Pidgin-heavy, utilitarian.\n"
        "aisha_kano: Kano teacher, Nigerian English with Muslim framing (Alhamdulillah).\n"
        "femi_abuja: Abuja banker, standard Nigerian English, individualist.\n"
        "ifeoma_ph: PH fashion + Nollywood fan, Nigerian English with film vocab.\n\n"
        "REGISTERS:\n"
        "nigerian_pidgin: 'abeg', 'no cap', 'wahala', 'e shock me', 'scatter', 'dey'.\n"
        "code_mixed: Mix English with Yoruba/Hausa/Igbo (ahn ahn, haba, nna, biko).\n"
        "nigerian_english: 'well done sir', 'no shaking', 'sharp sharp'.\n"
        "standard_english: Clear standard English, no slang.\n\n"
        "OUTPUT THE REVIEW ONLY:"
    ),
))

print("✅ Pipeline 1 configured")

In [ ]:
# Preview Pipeline 1
preview = data_designer.preview(config_builder=config_p1)
preview.display_sample_record()

In [ ]:
# Full Pipeline 1 run — resumable
df_p1 = generate_batched(
    config_builder=config_p1, total_records=P1_TARGET, batch_size=BATCH_SIZE,
    ckpt_dir=P1_CKPT_DIR, dataset_name="naija_p1_seed_jumia",
)
print(f"\n✅ Pipeline 1: {len(df_p1):,} records")

## 4. Pipeline 2 — Pure Synthetic: persona × product × register × stratified rating

In [ ]:
# Pipeline 2 config — seeded by Jumia product catalog
config_p2 = dd.DataDesignerConfigBuilder()
config_p2.add_model_config(MODEL_PRIMARY)
config_p2.with_seed_dataset(
    seed_source={"seed_type": "local", "path": str(SEED_PRODUCTS_PATH)},
    sampling_strategy="shuffle",
)

config_p2.add_column(dd.SamplerColumnConfig(
    name="persona_id", sampler_type=dd.SamplerType.CATEGORY,
    params=dd.CategorySamplerParams(values=["chinwe_owerri", "tunde_lagos", "aisha_kano", "femi_abuja", "ifeoma_ph"]),
))
config_p2.add_column(dd.SamplerColumnConfig(
    name="register_tier", sampler_type=dd.SamplerType.CATEGORY,
    params=dd.CategorySamplerParams(values=[
        "nigerian_pidgin", "nigerian_pidgin",
        "code_mixed", "code_mixed", "code_mixed",   # boosted in synthetic
        "nigerian_english", "standard_english",
    ]),
))
config_p2.add_column(dd.SamplerColumnConfig(
    name="target_rating", sampler_type=dd.SamplerType.CATEGORY,
    params=dd.CategorySamplerParams(values=["1", "2", "3", "4", "5"]),
))

config_p2.add_column(dd.LLMTextColumnConfig(
    name="persona_review", model_alias="naija-primary",
    system_prompt=(
        "You generate authentic Nigerian Jumia product reviews matching the user's persona voice, "
        "register tier, and target rating. Sound like a real Nigerian customer. Output ONLY the review."
    ),
    prompt=(
        "PRODUCT (Jumia):\n"
        "Title: {{ title }}\n"
        "Category: {{ category }}\n"
        "Description: {{ description }}\n"
        "Price: ₦{{ price_naira }}\n\n"
        "PERSONA: {{ persona_id }}\n"
        "REGISTER: {{ register_tier }}\n"
        "TARGET RATING: {{ target_rating }}/5\n\n"
        "Write a {{ target_rating }}/5 review (2-5 sentences) as this persona. Don't state the number — "
        "let the intensity show through tone.\n\n"
        "chinwe_owerri: Owerri Gen-Z, Igbo+English (nna, biko), communal-hedonic.\n"
        "tunde_lagos: Lagos trader, Pidgin-heavy, utilitarian.\n"
        "aisha_kano: Kano teacher, Nigerian English with Muslim framing.\n"
        "femi_abuja: Abuja banker, standard Nigerian English, individualist.\n"
        "ifeoma_ph: PH fashion + Nollywood fan, vibrant Nigerian English.\n\n"
        "nigerian_pidgin: 'abeg', 'no cap', 'wahala', 'e shock', 'scatter', 'dey'.\n"
        "code_mixed: Mix English w/ Yoruba/Hausa/Igbo (ahn ahn, haba, nna).\n"
        "nigerian_english: 'well done sir', 'no shaking', 'sharp sharp'.\n"
        "standard_english: Clear standard English.\n\n"
        "OUTPUT THE REVIEW ONLY:"
    ),
))

print("✅ Pipeline 2 configured")

In [ ]:
preview2 = data_designer.preview(config_builder=config_p2)
preview2.display_sample_record()

In [ ]:
df_p2 = generate_batched(
    config_builder=config_p2, total_records=P2_TARGET, batch_size=BATCH_SIZE,
    ckpt_dir=P2_CKPT_DIR, dataset_name="naija_p2_synthetic",
)
print(f"\n✅ Pipeline 2: {len(df_p2):,} records")

## 5. Combine + Pipeline 3 (LLM-as-Judge Quality Scoring)

In [ ]:
# Combine + dedup + quality filter (length)
def _to_unified(df, src):
    if len(df) == 0: return pd.DataFrame()
    out = pd.DataFrame()
    out["review"] = df["persona_review"]
    if src == "p1_seed":
        out["rating"] = pd.to_numeric(df["refined_rating"], errors="coerce").clip(1,5).fillna(3).astype(int)
        out["product_title"] = ""
        out["product_category"] = df.get("product_category", "")
    else:
        out["rating"] = pd.to_numeric(df["target_rating"], errors="coerce").clip(1,5).fillna(3).astype(int)
        out["product_title"] = df.get("title", "")
        out["product_category"] = df.get("category", "")
    out["persona_id"] = df["persona_id"]
    out["register_tier"] = df["register_tier"]
    out["source"] = src
    return out

df_combined = pd.concat([
    _to_unified(df_p1, "p1_seed"),
    _to_unified(df_p2, "p2_synthetic"),
], ignore_index=True)
df_combined = df_combined.dropna(subset=["review"])
df_combined = df_combined[df_combined["review"].str.len().between(50, 1500)]
df_combined = df_combined.drop_duplicates(subset=["review"], keep="first").reset_index(drop=True)

print(f"📊 combined: {len(df_combined):,}")
print(f"   by source:   {dict(df_combined['source'].value_counts())}")
print(f"   by register: {dict(df_combined['register_tier'].value_counts())}")
print(f"   by rating:   {dict(sorted(df_combined['rating'].value_counts().items()))}")

df_combined.to_parquet(CORPUS_DIR / "combined_pre_judge.parquet", index=False)

In [ ]:
# Pipeline 3 — LLM-as-Judge
judge_n = int(len(df_combined) * JUDGE_SAMPLE_PCT)
df_judge_seed = df_combined.sample(n=judge_n, random_state=42).reset_index(drop=True)
JUDGE_SEED_PATH = CORPUS_DIR / "judge_seed.parquet"
df_judge_seed.to_parquet(JUDGE_SEED_PATH, index=False)
print(f"⚖️ judging {judge_n:,} samples")

config_judge = dd.DataDesignerConfigBuilder()
config_judge.add_model_config(MODEL_JUDGE)
config_judge.with_seed_dataset(
    seed_source={"seed_type": "local", "path": str(JUDGE_SEED_PATH)},
    sampling_strategy="ordered",
)
config_judge.add_column(dd.LLMJudgeColumnConfig(
    name="quality", model_alias="naija-judge",
    prompt=(
        "Evaluate this Nigerian product review:\n\n"
        "PERSONA: {{ persona_id }}\n"
        "DECLARED REGISTER: {{ register_tier }}\n"
        "TARGET RATING: {{ rating }}/5\n"
        "REVIEW: {{ review }}"
    ),
    scores=[
        dd.Score(name="register_fidelity", description="Language matches the declared register tier?",
            options={"4":"Perfect natural markers", "3":"Good with minor drift",
                     "2":"Partial match", "1":"Wrong register", "0":"Not Nigerian"}),
        dd.Score(name="persona_authenticity", description="Sounds like the declared persona?",
            options={"4":"Captures voice precisely", "3":"Mostly authentic",
                     "2":"Partial", "1":"Generic", "0":"Contradicts persona"}),
        dd.Score(name="rating_coherence", description="Intensity matches the rating?",
            options={"4":"Perfect", "3":"Good", "2":"~1-star off", "1":"Strong mismatch", "0":"Contradicts"}),
        dd.Score(name="naturalness", description="Reads as real Nigerian (not AI)?",
            options={"4":"Indistinguishable from real", "3":"Mostly natural",
                     "2":"Some AI tells", "1":"Clearly AI", "0":"Robotic"}),
    ],
))

df_judged = generate_batched(
    config_builder=config_judge, total_records=judge_n, batch_size=BATCH_SIZE,
    ckpt_dir=P3_CKPT_DIR, dataset_name="naija_p3_judge",
)

score_cols = ["quality_register_fidelity", "quality_persona_authenticity",
              "quality_rating_coherence", "quality_naturalness"]
for c in score_cols:
    if c in df_judged.columns:
        df_judged[c] = pd.to_numeric(df_judged[c], errors="coerce")
avail = [c for c in score_cols if c in df_judged.columns]
if avail:
    df_judged["composite_score"] = df_judged[avail].mean(axis=1)
    print(f"\n📊 quality scores:")
    for c in avail:
        print(f"   {c:42s} mean={df_judged[c].mean():.2f}")
    print(f"   composite mean={df_judged['composite_score'].mean():.2f}")
    hq = df_judged[df_judged['composite_score'] >= QUALITY_THRESHOLD]
    print(f"   high-quality (≥{QUALITY_THRESHOLD}): {len(hq):,}/{len(df_judged):,} ({100*len(hq)/max(len(df_judged),1):.1f}%)")
df_judged.to_parquet(CORPUS_DIR / "judge_scored.parquet", index=False)

## 6. Final Filter + Stratified Split + Alpaca/ShareGPT Export

In [ ]:
from collections import defaultdict
import random as _r; _r.seed(42)

df_final = df_combined.copy()
if "composite_score" in df_judged.columns and len(df_judged):
    score_map = df_judged.set_index("review")["composite_score"].to_dict()
    df_final["composite_score"] = df_final["review"].map(score_map)
    keep = (df_final["composite_score"].isna()) | (df_final["composite_score"] >= QUALITY_THRESHOLD)
    n_before = len(df_final)
    df_final = df_final[keep].reset_index(drop=True)
    print(f"🔍 quality filter: {n_before:,} → {len(df_final):,}")

buckets = defaultdict(list)
for _, row in df_final.iterrows():
    buckets[row["register_tier"]].append(row.to_dict())

train, val, test = [], [], []
for tier, items in buckets.items():
    _r.shuffle(items)
    n = len(items)
    n_val, n_test = max(1, n//20), max(1, n//20)
    val   += items[:n_val]
    test  += items[n_val:n_val+n_test]
    train += items[n_val+n_test:]
_r.shuffle(train); _r.shuffle(val); _r.shuffle(test)
print(f"\nsplits: train={len(train):,}  val={len(val):,}  test={len(test):,}")

INSTRUCTION = (
    "Simulate the review behaviour of the following Nigerian user reviewing the described "
    "product. Generate the rating (1-5) and review text exactly as this user would write it. "
    "Match the user's register tier and cultural framing."
)

def to_alpaca(r):
    persona = json.dumps({"persona_id": r["persona_id"], "register_tier": r["register_tier"]}, ensure_ascii=False)
    product = json.dumps({"title": r.get("product_title",""), "category": r.get("product_category","")}, ensure_ascii=False)
    output  = json.dumps({"rating": int(r["rating"]), "review": r["review"]}, ensure_ascii=False)
    return {"instruction": INSTRUCTION,
            "input": f"### Persona\n{persona}\n\n### Product\n{product}\n\n### Register tier\n{r['register_tier']}",
            "output": output}

def to_sharegpt(r):
    user = f"{INSTRUCTION}\n\n### Persona\n{json.dumps({'persona_id': r['persona_id'], 'register_tier': r['register_tier']}, ensure_ascii=False)}\n\n### Product\n{json.dumps({'title': r.get('product_title',''), 'category': r.get('product_category','')}, ensure_ascii=False)}\n\n### Register tier\n{r['register_tier']}"
    asst = json.dumps({"rating": int(r["rating"]), "review": r["review"]}, ensure_ascii=False)
    return {"conversations": [{"from":"human","value":user}, {"from":"gpt","value":asst}]}

for name, rows in [("train", train), ("val", val), ("test", test)]:
    with jsonlines.open(CORPUS_DIR / f"v1_{name}_alpaca.jsonl", mode="w") as w:
        for r in rows: w.write(to_alpaca(r))
    with jsonlines.open(CORPUS_DIR / f"v1_{name}_sharegpt.jsonl", mode="w") as w:
        for r in rows: w.write(to_sharegpt(r))
    pd.DataFrame(rows).to_parquet(CORPUS_DIR / f"v1_{name}_full.parquet", index=False)
    print(f"  💾 {name}: {len(rows):,} → alpaca + sharegpt + full parquet")

all_rows = train+val+test
summary = {
    "train": len(train), "val": len(val), "test": len(test), "total": len(all_rows),
    "by_source": dict(pd.DataFrame(all_rows)["source"].value_counts()),
    "by_register": dict(pd.DataFrame(all_rows)["register_tier"].value_counts()),
    "by_rating": dict(sorted(pd.DataFrame(all_rows)["rating"].value_counts().items())),
}
(CORPUS_DIR / "v1_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(f"\n✅ {summary['total']:,} records")
print(json.dumps(summary, indent=2, default=str))

## 7. Unsloth QLoRA Fine-Tune

In [ ]:
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))
print("✅ HF login")

In [ ]:
import yaml
with open("finetuning/configs/naija_reviewer_qlora.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["per_device_train_batch_size"] = GPU_CFG["batch"]
cfg["per_device_eval_batch_size"]  = GPU_CFG["batch"]
cfg["gradient_accumulation_steps"] = GPU_CFG["grad_accum"]
cfg["max_seq_length"] = GPU_CFG["seq_len"]
cfg["bf16"] = BF16_OK; cfg["fp16"] = not BF16_OK
cfg["output_dir"] = str(TRAINING_CKPT)
cfg["save_steps"] = 200; cfg["eval_steps"] = 200
cfg["save_strategy"] = "steps"; cfg["evaluation_strategy"] = "steps"
cfg["save_total_limit"] = 3; cfg["logging_steps"] = 25

checkpoints = sorted(TRAINING_CKPT.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
RESUME_FROM = checkpoints[-1] if checkpoints else None
print(f"resume: {RESUME_FROM.name if RESUME_FROM else 'fresh start'}")

In [ ]:
from unsloth import FastLanguageModel
if RESUME_FROM:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(RESUME_FROM), max_seq_length=cfg["max_seq_length"],
        dtype=None, load_in_4bit=True,
    )
    FastLanguageModel.for_training(model)
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
        max_seq_length=cfg["max_seq_length"], dtype=None, load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=cfg["lora_r"], target_modules=cfg["lora_target_modules"],
        lora_alpha=cfg["lora_alpha"], lora_dropout=cfg["lora_dropout"],
        bias="none", use_gradient_checkpointing="unsloth", random_state=cfg["seed"],
    )
tokenizer.padding_side = "right"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print(f"✅ trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M")

In [ ]:
# Load Alpaca data
from datasets import Dataset
def alpaca_to_text(r):
    return {"text": f"### Instruction\n{r['instruction']}\n\n### Input\n{r['input']}\n\n### Response\n{r['output']}"}
ds_train_raw = [json.loads(l) for l in (CORPUS_DIR / "v1_train_alpaca.jsonl").open()]
ds_val_raw   = [json.loads(l) for l in (CORPUS_DIR / "v1_val_alpaca.jsonl").open()]
ds_train = Dataset.from_list(ds_train_raw).map(alpaca_to_text)
ds_val   = Dataset.from_list(ds_val_raw).map(alpaca_to_text)
print(f"train={len(ds_train):,}  val={len(ds_val):,}")

In [ ]:
import wandb
try:
    wandb.init(project="naija-persona-agent", name=cfg["run_name"],
               config={**cfg, "gpu": GPU_NAME, "framework": "data_designer+unsloth"},
               mode="online", resume="allow")
except Exception as e:
    print(f"⚠️ W&B: {e}"); os.environ["WANDB_DISABLED"] = "true"

In [ ]:
from trl import SFTTrainer, SFTConfig
import time

training_args = SFTConfig(
    output_dir=cfg["output_dir"], num_train_epochs=cfg["num_train_epochs"],
    per_device_train_batch_size=cfg["per_device_train_batch_size"],
    per_device_eval_batch_size=cfg["per_device_eval_batch_size"],
    gradient_accumulation_steps=cfg["gradient_accumulation_steps"],
    learning_rate=cfg["learning_rate"], lr_scheduler_type=cfg["lr_scheduler_type"],
    warmup_steps=cfg["warmup_steps"], weight_decay=cfg["weight_decay"],
    bf16=cfg["bf16"], fp16=cfg["fp16"], optim=cfg["optim"],
    logging_steps=cfg["logging_steps"],
    save_strategy="steps", save_steps=cfg["save_steps"],
    eval_strategy="steps", eval_steps=cfg["eval_steps"],
    save_total_limit=cfg["save_total_limit"], seed=cfg["seed"],
    report_to=["wandb"] if not os.environ.get("WANDB_DISABLED") else ["none"],
    run_name=cfg["run_name"], max_length=cfg["max_seq_length"],
    packing=False, dataset_text_field="text",
    save_safetensors=True, tf32=True, group_by_length=True,
)
trainer = SFTTrainer(model=model, train_dataset=ds_train, eval_dataset=ds_val,
                     tokenizer=tokenizer, args=training_args)

print("🚀 training...")
t0 = time.time()
trainer.train(resume_from_checkpoint=str(RESUME_FROM)) if RESUME_FROM else trainer.train()
print(f"✅ done in {(time.time()-t0)/60:.1f}min")

FINAL_ADAPTER = TRAINING_CKPT / "final-adapter"
model.save_pretrained(str(FINAL_ADAPTER))
tokenizer.save_pretrained(str(FINAL_ADAPTER))
print(f"✅ adapter → {FINAL_ADAPTER}")

In [ ]:
# Merge LoRA via Unsloth one-liner
import gc
if (MERGED_DIR / "config.json").exists():
    print("✅ merged exists — skipping")
else:
    model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")
    print(f"✅ merged → {MERGED_DIR}")
    del trainer, model; gc.collect(); torch.cuda.empty_cache()

## 8. Smoke Test + HuggingFace Push

In [ ]:
from unsloth import FastLanguageModel
naija_model, naija_tok = FastLanguageModel.from_pretrained(
    model_name=str(MERGED_DIR), max_seq_length=GPU_CFG["seq_len"], dtype=None, load_in_4bit=False,
)
FastLanguageModel.for_inference(naija_model)

ds_test_raw = [json.loads(l) for l in (CORPUS_DIR / "v1_test_alpaca.jsonl").open()]
s = ds_test_raw[0]
prompt = f"### Instruction\n{s['instruction']}\n\n### Input\n{s['input']}\n\n### Response\n"
inputs = naija_tok(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    out = naija_model.generate(**inputs, max_new_tokens=200, temperature=0.7,
                                do_sample=True, pad_token_id=naija_tok.eos_token_id)
result = naija_tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
print(f"GT:    {s['output']}")
print(f"\nMODEL: {result[:400]}")

In [ ]:
from huggingface_hub import create_repo, upload_folder
HF_ORG = "Mystique1337"
MODEL_REPO   = f"{HF_ORG}/naija-reviewer-8b"
ADAPTER_REPO = f"{HF_ORG}/naija-reviewer-8b-lora"
DATASET_REPO = f"{HF_ORG}/npa-corpus-v1"

for repo, kind in [(MODEL_REPO,"model"), (ADAPTER_REPO,"model"), (DATASET_REPO,"dataset")]:
    try: create_repo(repo, repo_type=kind, exist_ok=True, private=True); print(f"  ✅ {repo}")
    except Exception as e: print(f"  ⚠️ {repo}: {e}")

upload_folder(folder_path=str(MERGED_DIR), repo_id=MODEL_REPO, repo_type="model")
upload_folder(folder_path=str(FINAL_ADAPTER), repo_id=ADAPTER_REPO, repo_type="model",
              ignore_patterns=["merged/*", "checkpoint-*", "runs/*"])
upload_folder(folder_path=str(CORPUS_DIR), repo_id=DATASET_REPO, repo_type="dataset",
              ignore_patterns=["raw_*", "checkpoints/*"])
print("\n🎉 uploads done (private — flip public when ready)")

## 9. GGUF Conversion for Ollama (local)

On your laptop after this notebook:

```bash
git clone https://github.com/ggerganov/llama.cpp && cd llama.cpp && make
huggingface-cli download Mystique1337/naija-reviewer-8b --local-dir ./merged
python3 convert_hf_to_gguf.py ./merged --outfile naija-reviewer-8b-f16.gguf --outtype f16
./llama-quantize naija-reviewer-8b-f16.gguf naija-reviewer-8b-Q4_K_M.gguf Q4_K_M
cp naija-reviewer-8b-Q4_K_M.gguf telcoproject/finetuning/
cd telcoproject && ollama create naija-reviewer-8b -f finetuning/Modelfile
```

Then `echo 'TASK1_BACKBONE=ollama:naija-reviewer-8b' >> .env && make demo`.

## 🔧 Utility — Reset Pipeline Checkpoints

Uncomment **only** if you want to regenerate a pipeline from scratch.

In [ ]:
# import shutil
# shutil.rmtree(P1_CKPT_DIR, ignore_errors=True); P1_CKPT_DIR.mkdir(parents=True, exist_ok=True); print("🗑️ P1 wiped")
# shutil.rmtree(P2_CKPT_DIR, ignore_errors=True); P2_CKPT_DIR.mkdir(parents=True, exist_ok=True); print("🗑️ P2 wiped")
# shutil.rmtree(P3_CKPT_DIR, ignore_errors=True); P3_CKPT_DIR.mkdir(parents=True, exist_ok=True); print("🗑️ P3 wiped")